# 2. Router 평가

학습과 threshold 보정에 사용하지 않은 project-disjoint test split을 한 번만 평가합니다. Full-5, 고정 Top-2, Utility Top-2, 확률식 escalation, 학습 gate를 같은 outcome matrix에서 비교합니다.

In [1]:
import json
import sys
from pathlib import Path
from pprint import pprint

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
DATA_DIR = ROOT / 'data' / 'phase2e_combined'
ARTIFACT_DIR = ROOT / 'artifacts' / 'phase2e'
RESULTS_DIR = ROOT / 'results' / 'router'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

from llm_security.datasets import load_router_samples_jsonl, load_utility_samples_jsonl
from llm_security.models import ExpertFamily, to_dict
from llm_security.routing import AnchorRareRouter, BudgetedUtilityRouter
from llm_security.experiments import write_utility_tradeoff_report

In [2]:
anchor_path = ARTIFACT_DIR / 'router_anchor_rare_v2.pkl'
test_path = DATA_DIR / 'semantic' / 'router_test.jsonl'
if not anchor_path.exists():
    raise FileNotFoundError('먼저 01_train_router.ipynb를 실행하세요.')
router = AnchorRareRouter.load(anchor_path)
test_samples = load_router_samples_jsonl(test_path)
anchor_metrics = router.evaluate(test_samples)

fixed_anchors = {ExpertFamily.MEMORY_BOUNDS, ExpertFamily.CONTROL_STATE_ERROR}
fixed_exact_coverage = sum(
    set(sample.labels).issubset(fixed_anchors) for sample in test_samples
) / len(test_samples)
summary = {
    'artifact': str(anchor_path),
    'fixed_top2_exact_coverage': fixed_exact_coverage,
    'anchor_rare': to_dict(anchor_metrics),
}
pprint(summary)

{'anchor_rare': {'average_experts_per_candidate': 2.920255183413078,
                 'exact_coverage': 0.9920255183413078,
                 'expert_coverage': 0.9920255183413078,
                 'llm_calls_saved_vs_all_six': 1931,
                 'rare_family_metrics': {'concurrency_toctou': {'false_positive': 0,
                                                                'positive': 10,
                                                                'precision': 1.0,
                                                                'recall': 1.0,
                                                                'threshold': 0.85,
                                                                'trigger_rate': 0.01594896331738437,
                                                                'triggered': 10,
                                                                'true_positive': 10},
                                         'integer_size_type': {'false_positive': 13,
     

In [3]:
utility_path = ARTIFACT_DIR / 'router_top2_full5_v4.pkl'
utility_test_path = ROOT / 'data' / 'utility' / 'outcomes_test.jsonl'
if utility_path.exists() and utility_test_path.exists():
    utility_router = BudgetedUtilityRouter.load(utility_path)
    utility_test = load_utility_samples_jsonl(utility_test_path)
    utility_metrics = utility_router.evaluate_baselines(
        utility_test, anchor_router=router
    )
    summary['utility'] = to_dict(utility_metrics)
else:
    summary['utility'] = {
        'evaluated': False,
        'reason': 'Utility artifact or outcomes_test.jsonl not found',
    }
pprint(summary['utility'])

{'evaluated': False,
 'reason': 'Utility artifact or outcomes_test.jsonl not found'}


## 해석 기준

truth_recall과 exact_coverage를 먼저 확인합니다. 목표 recall을 만족하는 정책끼리 average_assignments, full5_rate, token, cost, latency를 비교합니다. missed_escalation_rate는 Full-5가 필요한데 Top-2로 끝낸 비율이며 가장 중요한 Gate 실패 지표입니다. Expert success predictor와 Gate의 Brier/ECE는 별도 필드로 확인합니다. family Top-1 accuracy는 최적화 목표가 아닙니다.

In [4]:
metrics_path = RESULTS_DIR / 'router_metrics.json'
metrics_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print('saved:', metrics_path)
if isinstance(summary.get('utility'), dict) and 'evaluated' not in summary['utility']:
    figures = write_utility_tradeoff_report(
        summary['utility'], RESULTS_DIR / 'utility_figures'
    )
    pprint(figures)

saved: C:\Users\junhyun111\Desktop\llm-security\results\router\router_metrics.json
